# Experiment description
## Goal
Goal is to test the hypothesis that the addition of an uncertainty metric based on the depth "landscape" surrounding a keypoint to the already present baseline keypoint location uncertainty (in the case of orbslam, octave-based weighing) leads to improvement in SLAM trajectory estimation.
## Setting
### Infrastructure
Using pyslam with the following tweaks:
- Containerized into a Singularity image and launched on the Mila cluster.
- `USE_CPP_CORE = False` : all runs use the Python core (C++ backend compiled but not active). Relative comparisons across presets are valid, but absolute numbers may differ from C++ runs.
- Deterministic ORB extraction enabled (`ORBextractorDeterministic`), cuDNN determinism flags on.
- Tracking-lost counter includes both LOST and RELOCALIZE frames —> `percent_lost` is slightly broader than upstream definition.
### Dataset
TartanAir-v2, "easy" sequences only, 18 environments, 640×480 resolution, synthetic ground truth depth maps.
### Method
The depth-weighting function `_compute_kps_depth_weight` in `frame.py`:
1. Loads an auxiliary depth map (ground truth or Depth Anything V2 estimate).
2. Clips depth > 20m to NaN (sky cutoff).
3. Computes squared Sobel gradient magnitude: $\sigma^2_d = \|\nabla D\|^2$ (ksize=5).
4. Samples $\sigma^2_d$ at keypoint locations.
5. Computes per-keypoint weight $w = 1/(1 + \lambda \cdot \sigma^2_d)$.
6. **Mean-1 normalizes** over valid-depth keypoints —> this redistributes rather than globally loosens influence, avoiding a confound with the chi²-threshold outlier rejection 
7. Keypoints with invalid depth fall back to weight 1.0.

Weights are applied by multiplying `invSigma2` at four optimizer call sites (BA, pose optimization, local BA, LBA process).
### Presets
- `baseline`: octave-based keypoint uncertainty only.
- `depth_gt_{1,2,3}`: baseline × depth weight with $\lambda \in \{0.01, 0.1, 1.0\}$, $D$ = ground truth depth.
- `depth_est_{1,2,3}`: same, but $D$ = Depth Anything V2 estimate.
- Each (environment, preset, sequence) combination is run with 10 times.
## What we're looking to see
Clear evidence that adding depth-structure-derived uncertainty to BA in SLAM improves trajectory estimation, the challenge being for it to be statistically meaningful given the highly non-deterministic nature of a system such as ORB-SLAM on a dataset like TartanAir.